# 04 — Evaluate ModernBERT on CSFCube

Each query paper and candidate paper is represented as title plus
abstract, matching the existing CSFCube notebook. The model scores
every released candidate exactly once and writes the file names
expected by `ranking_eval.py` for background, method, and result.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "modernbert_supervised_baseline.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned repository; "
        "tools/modernbert_supervised_baseline.py was not found."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root: {REPO_ROOT}")

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers.cross_encoder import CrossEncoder

from tools.modernbert_supervised_baseline import (
    build_csfcube_rankings,
    read_jsonl,
    reproducibility_manifest,
    sha256_file,
    write_json,
)

ARTIFACT_DIR = Path(os.environ.get(
    "MODERNBERT_ARTIFACT_DIR",
    str(REPO_ROOT / "artifacts" / "modernbert_supervised_baseline"),
))
MODEL_DIR = ARTIFACT_DIR / "models" / "modernbert_base_dorismae_lambda_seed3407" / "final"
CSFCUBE_DIR = REPO_ROOT / "CSFCube"
PAPER_PATH = CSFCUBE_DIR / "abstracts-csfcube-preds.jsonl"
OUTPUT_DIR = ARTIFACT_DIR / "csfcube"
EXPERIMENT = "modernbert-dorismae-lambda-seed3407"
FACETS = ("background", "method", "result")
MAX_LENGTH = 1024
PREDICT_BATCH_SIZE = 64

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Run notebook 02 first; missing {MODEL_DIR}")
if not PAPER_PATH.exists():
    raise FileNotFoundError(PAPER_PATH)
if not hasattr(np, "asfarray"):
    raise RuntimeError(
        "The unchanged CSFCube rank_metrics.py requires NumPy 1.26.4; "
        f"the active kernel has NumPy {np.__version__}. Run "
        "`python -m pip install numpy==1.26.4`, restart the kernel, "
        "and rerun this notebook."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
paper_rows = read_jsonl(PAPER_PATH)
paper_lookup = {}
for item in paper_rows:
    abstract = item.get("abstract", "")
    if isinstance(abstract, list):
        abstract = " ".join(str(sentence) for sentence in abstract)
    paper_lookup[int(item["paper_id"])] = (
        f"Title: {item.get('title', '')}\nAbstract: {abstract}"
    )

model = CrossEncoder(str(MODEL_DIR), max_length=MAX_LENGTH)

def score_pairs(pairs):
    return model.predict(
        pairs,
        batch_size=PREDICT_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

print(f"Paper texts: {len(paper_lookup):,}")

In [ ]:
run_records = {}
for facet in FACETS:
    gold_path = CSFCUBE_DIR / f"test-pid2anns-csfcube-{facet}.json"
    annotations = json.loads(gold_path.read_text(encoding="utf-8"))
    expected_sets = {
        str(query_id): {str(value) for value in item["cands"]}
        for query_id, item in annotations.items()
    }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.perf_counter()
    rankings = build_csfcube_rankings(annotations, paper_lookup, score_pairs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    assert set(rankings) == set(expected_sets)
    assert all(
        {item[0] for item in rankings[query_id]} == expected
        for query_id, expected in expected_sets.items()
    )
    output_path = OUTPUT_DIR / (
        f"test-pid2pool-csfcube-{EXPERIMENT}-{facet}-ranked.json"
    )
    write_json(output_path, rankings)
    document_count = sum(len(value) for value in rankings.values())
    run_records[facet] = {
        "gold_path": str(gold_path.resolve()),
        "gold_sha256": sha256_file(gold_path),
        "output": str(output_path.resolve()),
        "output_sha256": sha256_file(output_path),
        "queries": len(rankings),
        "documents": document_count,
        "elapsed_seconds": elapsed,
        "queries_per_second": len(rankings) / elapsed,
        "documents_per_second": document_count / elapsed,
        "peak_vram": (
            torch.cuda.max_memory_reserved() / (1024 ** 3)
            if torch.cuda.is_available() else 0.0
        ),
    }
    print(facet, json.dumps(run_records[facet], indent=2))

In [ ]:
# Save evaluator stdout because ranking_eval.py reports, rather than serializes, metrics.
for facet in (*FACETS, "all"):
    command = [
        sys.executable,
        str(CSFCUBE_DIR / "eval_scripts" / "ranking_eval.py"),
        "eval_pool_ranking",
        "--gold_path",
        str(CSFCUBE_DIR),
        "--ranked_path",
        str(OUTPUT_DIR),
        "--experiment",
        EXPERIMENT,
        "--facet",
        facet,
    ]
    print("Running:", " ".join(command))
    result = subprocess.run(command, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("CSFCube evaluator stderr:")
        print(result.stderr)
        raise RuntimeError(
            f"CSFCube evaluator failed for facet {facet!r} with "
            f"exit code {result.returncode}. See stderr above."
        )
    (OUTPUT_DIR / f"metrics-{facet}.txt").write_text(
        result.stdout,
        encoding="utf-8",
    )

manifest = {
    "experiment": EXPERIMENT,
    "model_directory": str(MODEL_DIR.resolve()),
    "query_representation": "query-paper title plus full abstract",
    "candidate_representation": "candidate-paper title plus full abstract",
    "max_length": MAX_LENGTH,
    "predict_batch_size": PREDICT_BATCH_SIZE,
    "runs": run_records,
    "environment": reproducibility_manifest(),
}
write_json(OUTPUT_DIR / "csfcube_evaluation_manifest.json", manifest)

Retain all three ranked JSON files. They are the per-query data needed
for the clustered paired bootstrap at unique-paper level; the printed
aggregate nDCG%20 values are not sufficient for paired uncertainty.